# SMISDroid — MobileBERT Fine-Tuning for SMS Fraud Detection

This notebook fine-tunes **MobileBERT** on your `train.csv` dataset and exports a quantized **TFLite** model for on-device inference in the SMISDroid Flutter app.

### Local Setup (RTX 2050)
1. Select kernel: **venv** (`model_training/venv/bin/python`)
2. Run all cells top to bottom
3. Output files are saved directly to `assets/`

---
## Step 1: Check GPU & Install Dependencies

In [ ]:
# Check GPU
import subprocess
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print("nvidia-smi not found, checking TF GPU detection instead...")

In [ ]:
# Dependencies already installed in venv — just verify
import tensorflow as tf
print(f"TensorFlow: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs found: {gpus}")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("GPU memory growth enabled")
else:
    print("WARNING: No GPU detected — training will be slow on CPU")

---
## Step 2: Upload & Prepare Dataset

Your dataset `train.csv` has columns: `message_text`, `class_label`

Labels: `benign`, `kyc_scam`, `impersonation`, `phishing_link`, `fake_payment_portal`, `account_block_scam`

We map: **benign → 0 (safe)** | **everything else → 1 (fraud)**

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

os.makedirs("output", exist_ok=True)

In [ ]:
# Load train.csv from local datasets folder
filename = "datasets/train.csv"
print(f"Loading: {filename}")

In [ ]:
# Load and clean train.csv
raw_df = pd.read_csv(filename)
print(f"Raw rows: {len(raw_df)}")
print(f"Columns: {list(raw_df.columns)}")

# Keep only valid rows with proper labels
VALID_LABELS = {"benign", "kyc_scam", "impersonation", "phishing_link", "fake_payment_portal", "account_block_scam"}
df = raw_df[["message_text", "class_label"]].dropna()
df = df[df["class_label"].isin(VALID_LABELS)].copy()

# Map labels: benign=0, all fraud types=1
df["label"] = (df["class_label"] != "benign").astype(int)
df = df.rename(columns={"message_text": "text"})

print(f"\nCleaned rows: {len(df)}")
print(f"\nLabel distribution:")
print(df["class_label"].value_counts())
print(f"\nBinary: Fraud={df['label'].sum()} ({df['label'].mean()*100:.1f}%) | Safe={(df['label']==0).sum()} ({(df['label']==0).mean()*100:.1f}%)")
df[["text", "label"]].head()

In [ ]:
# Final dataset summary
combined = df[["text", "label"]].drop_duplicates(subset=["text"]).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"{'='*50}")
print(f"  FINAL DATASET")
print(f"{'='*50}")
print(f"  Total:  {len(combined)}")
print(f"  Fraud:  {combined['label'].sum()} ({combined['label'].mean()*100:.1f}%)")
print(f"  Safe:   {(combined['label']==0).sum()} ({(combined['label']==0).mean()*100:.1f}%)")
print(f"{'='*50}")

In [ ]:
# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    combined["text"].tolist(),
    combined["label"].tolist(),
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=combined["label"]
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Train fraud ratio: {sum(y_train)/len(y_train)*100:.1f}%")
print(f"Test fraud ratio:  {sum(y_test)/len(y_test)*100:.1f}%")

---
## Step 3: Tokenize with MobileBERT Tokenizer

In [ ]:
from transformers import MobileBertTokenizer

MODEL_NAME = "google/mobilebert-uncased"
MAX_SEQ_LENGTH = 128

tokenizer = MobileBertTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")

# Quick test
sample = "Your account is blocked. Verify at http://verify.xyz"
tokens = tokenizer(sample, max_length=MAX_SEQ_LENGTH, padding="max_length", truncation=True)
print(f"\nSample tokenization:")
print(f"  Input: {sample}")
print(f"  Token IDs (first 20): {tokens['input_ids'][:20]}")
print(f"  Decoded: {tokenizer.decode(tokens['input_ids'][:20])}")

In [ ]:
# Tokenize all data
print("Tokenizing train set...")
train_encodings = tokenizer(
    X_train,
    max_length=MAX_SEQ_LENGTH,
    padding="max_length",
    truncation=True,
    return_tensors="tf"
)

print("Tokenizing test set...")
test_encodings = tokenizer(
    X_test,
    max_length=MAX_SEQ_LENGTH,
    padding="max_length",
    truncation=True,
    return_tensors="tf"
)

print(f"Train input_ids shape: {train_encodings['input_ids'].shape}")
print(f"Test input_ids shape:  {test_encodings['input_ids'].shape}")

In [ ]:
# Create TF datasets
BATCH_SIZE = 16

train_dataset = tf.data.Dataset.from_tensor_slices((
    {
        "input_ids": train_encodings["input_ids"],
        "attention_mask": train_encodings["attention_mask"],
        "token_type_ids": train_encodings["token_type_ids"],
    },
    np.array(y_train)
)).shuffle(len(y_train), seed=RANDOM_SEED).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_tensor_slices((
    {
        "input_ids": test_encodings["input_ids"],
        "attention_mask": test_encodings["attention_mask"],
        "token_type_ids": test_encodings["token_type_ids"],
    },
    np.array(y_test)
)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(f"Train batches: {len(train_dataset)}")
print(f"Test batches:  {len(test_dataset)}")

---
## Step 4: Fine-Tune MobileBERT

In [ ]:
from transformers import TFMobileBertForSequenceClassification

# Load pre-trained MobileBERT with classification head
model = TFMobileBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2  # 0=legitimate, 1=fraud
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Total parameters: {model.count_params():,}")

In [ ]:
# Compile
EPOCHS = 3
LEARNING_RATE = 2e-5

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

print(f"Optimizer: Adam (lr={LEARNING_RATE})")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")

In [ ]:
# Train!
print("\n" + "="*60)
print("  TRAINING STARTED — This takes ~15-30 min on T4 GPU")
print("="*60 + "\n")

history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=EPOCHS,
    verbose=1
)

print("\n" + "="*60)
print("  TRAINING COMPLETE")
print("="*60)

---
## Step 5: Evaluate Model

In [ ]:
# Get predictions
predictions = model.predict(test_dataset)
y_pred = np.argmax(predictions.logits, axis=-1)

# Classification Report
print("\n" + "="*60)
print("  CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_test, y_pred, target_names=["Legitimate", "Fraud"]))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print(f"Confusion Matrix:")
print(f"                  Predicted Legit  Predicted Fraud")
print(f"  Actually Legit:    TN={cm[0][0]:>5}        FP={cm[0][1]:>5}")
print(f"  Actually Fraud:    FN={cm[1][0]:>5}        TP={cm[1][1]:>5}")

accuracy = (cm[0][0] + cm[1][1]) / cm.sum()
print(f"\nOverall Accuracy: {accuracy*100:.2f}%")

In [ ]:
# Test on sample messages
test_messages = [
    "Your electricity bill Rs.2500 due immediately! Pay at http://bill-pay.xyz or disconnection TODAY!",
    "Dear Customer, Rs.5000 debited from A/c XX1234. Avl Bal Rs.25000. -ICICI",
    "URGENT: Your HDFC account blocked. Verify at http://hdfc-verify.tk",
    "Your Swiggy order is on the way. Delivery in 25 minutes.",
    "KYC expired! Update now at http://kyc-update.cf or account will be suspended",
    "Meeting at 3pm tomorrow. See you there.",
    "You won Rs.50000 in lottery! Claim at http://claim-prize.ga before it expires",
    "OTP for transaction: 847291. Valid for 5 min. Do not share. -SBI",
]

print("\n" + "="*80)
print("  LIVE TEST ON SAMPLE MESSAGES")
print("="*80)

for msg in test_messages:
    tokens = tokenizer(msg, max_length=MAX_SEQ_LENGTH, padding="max_length", truncation=True, return_tensors="tf")
    output = model(tokens)
    probs = tf.nn.softmax(output.logits, axis=-1).numpy()[0]
    fraud_prob = probs[1]
    label = "FRAUD" if fraud_prob > 0.5 else "SAFE"
    icon = "\u274c" if label == "FRAUD" else "\u2705"
    print(f"\n{icon} [{label}] Score: {fraud_prob:.4f}")
    print(f"   {msg[:80]}{'...' if len(msg)>80 else ''}")

---
## Step 6: Export to TFLite (Quantized)

In [ ]:
# Create a wrapper that takes only input_ids and outputs fraud probability
class TFLiteExportModel(tf.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    @tf.function(input_signature=[
        tf.TensorSpec(shape=[1, MAX_SEQ_LENGTH], dtype=tf.int32, name="input_ids")
    ])
    def predict(self, input_ids):
        attention_mask = tf.cast(tf.not_equal(input_ids, 0), tf.int32)
        token_type_ids = tf.zeros_like(input_ids)
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            training=False
        )
        probs = tf.nn.softmax(outputs.logits, axis=-1)
        return probs[:, 1:2]  # Return only fraud probability

export_model = TFLiteExportModel(model)

# Verify wrapper works
test_input = tokenizer("Test message", max_length=MAX_SEQ_LENGTH, padding="max_length", truncation=True, return_tensors="tf")
result = export_model.predict(test_input["input_ids"])
print(f"Wrapper test — fraud probability: {result.numpy()[0][0]:.4f}")

In [ ]:
# Convert to TFLite with dynamic range quantization
print("Converting to TFLite (this may take a few minutes)...")

concrete_func = export_model.predict.get_concrete_function()

converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # Dynamic range quantization
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS     # Needed for some transformer ops
]
converter._experimental_lower_tensor_list_ops = False

tflite_model = converter.convert()

# Save
tflite_path = "output/fraud_model.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

size_mb = len(tflite_model) / 1024 / 1024
print(f"\nModel saved: {tflite_path}")
print(f"Model size: {size_mb:.1f} MB")

---
## Step 7: Verify TFLite Model

In [ ]:
# Load and test TFLite model
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("TFLite Model Info:")
print(f"  Input:  {input_details[0]['name']} — shape {input_details[0]['shape']} dtype {input_details[0]['dtype']}")
print(f"  Output: {output_details[0]['name']} — shape {output_details[0]['shape']} dtype {output_details[0]['dtype']}")

In [ ]:
# Accuracy test on full test set
print("Running TFLite accuracy test on test set...")

n_test = len(X_test)
correct = 0
tflite_preds = []

for i in range(n_test):
    tokens = tokenizer(
        X_test[i],
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="np"
    )
    interpreter.set_tensor(input_details[0]["index"], tokens["input_ids"].astype(np.int32))
    interpreter.invoke()
    score = interpreter.get_tensor(output_details[0]["index"])[0][0]
    pred = 1 if score > 0.5 else 0
    tflite_preds.append(pred)
    if pred == y_test[i]:
        correct += 1
    if (i+1) % 200 == 0:
        print(f"  Tested {i+1}/{n_test} — Running accuracy: {correct/(i+1)*100:.1f}%")

print(f"\n{'='*60}")
print(f"  TFLite Model Accuracy: {correct}/{n_test} ({correct/n_test*100:.2f}%)")
print(f"{'='*60}")
print("\nTFLite Classification Report:")
print(classification_report(y_test, tflite_preds, target_names=["Legitimate", "Fraud"]))

In [ ]:
# Live test with TFLite
print("\n" + "="*80)
print("  TFLite LIVE TEST")
print("="*80)

for msg in test_messages:
    tokens = tokenizer(msg, max_length=MAX_SEQ_LENGTH, padding="max_length", truncation=True, return_tensors="np")
    interpreter.set_tensor(input_details[0]["index"], tokens["input_ids"].astype(np.int32))
    interpreter.invoke()
    score = interpreter.get_tensor(output_details[0]["index"])[0][0]
    label = "FRAUD" if score > 0.5 else "SAFE"
    icon = "\u274c" if label == "FRAUD" else "\u2705"
    print(f"\n{icon} [{label}] Score: {score:.4f}")
    print(f"   {msg[:80]}{'...' if len(msg)>80 else ''}")

---
## Step 8: Export Flutter Assets

In [ ]:
# Export vocabulary.json
vocab = tokenizer.get_vocab()
sorted_vocab = {k: int(v) for k, v in sorted(vocab.items(), key=lambda x: x[1])}

vocab_path = "output/vocabulary.json"
with open(vocab_path, "w") as f:
    json.dump(sorted_vocab, f)
print(f"Vocabulary: {vocab_path} ({len(sorted_vocab)} tokens)")

# Export nlp_config.json
config = {
    "model_type": "mobilebert",
    "model_name": MODEL_NAME,
    "max_seq_length": MAX_SEQ_LENGTH,
    "vocab_size": tokenizer.vocab_size,
    "input_type": "token_ids",
    "output_type": "fraud_probability",
    "threshold_safe": 0.3,
    "threshold_suspicious": 0.6,
    "pad_token_id": tokenizer.pad_token_id,
    "cls_token_id": tokenizer.cls_token_id,
    "sep_token_id": tokenizer.sep_token_id,
    "unk_token_id": tokenizer.unk_token_id,
    "version": "1.0.0",
    "training": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "train_samples": len(X_train),
        "test_samples": len(X_test),
        "test_accuracy": round(correct / n_test, 4),
    }
}

config_path = "output/nlp_config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
print(f"Config: {config_path}")

# Export model_weights.json (metadata only — actual weights are in .tflite)
weights_meta = {
    "type": "mobilebert",
    "base_model": MODEL_NAME,
    "total_params": int(model.count_params()),
    "quantization": "dynamic_range",
    "tflite_size_bytes": len(tflite_model),
}

weights_path = "output/model_weights.json"
with open(weights_path, "w") as f:
    json.dump(weights_meta, f, indent=2)
print(f"Weights metadata: {weights_path}")

print(f"\n{'='*60}")
print(f"  ALL FILES EXPORTED TO output/ FOLDER")
print(f"{'='*60}")
!ls -lh output/

---
## Step 9: Copy to Flutter Assets

Files are copied directly to `assets/` in your project. Then rebuild with `flutter build apk --debug`.

In [ ]:
import shutil

copies = [
    ("output/fraud_model.tflite", "../assets/models/fraud_model.tflite"),
    ("output/vocabulary.json", "../assets/nlp/vocabulary.json"),
    ("output/nlp_config.json", "../assets/nlp/nlp_config.json"),
    ("output/model_weights.json", "../assets/nlp/model_weights.json"),
]

print("Copying to Flutter assets...\n")
for src, dst in copies:
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(src, dst)
    size = os.path.getsize(dst)
    print(f"  {dst} ({size/1024:.1f} KB)")

print("\nDone! Run: flutter build apk --debug")